In [16]:
import psycopg2
import pandas as pd
import numpy as np
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ======================
# CONFIG
# ======================
TOP_N = 200   # 🔥 tăng recall
TOP_K = 5
DB_CONFIG = {
    "host": "aws-1-ap-northeast-2.pooler.supabase.com",  # 🔥 đổi lại
    "port": 6543,                          # 🔥 không dùng 6543 nữa
    "database": "postgres",
    "user": "postgres.ypkwqsbsunlvpoqdadbq",
    "password": "5$eAK8EV4S+gsKj",
    "sslmode": "require"
}
# ======================
# USER CONTEXT
# ======================
user_context = {
    "user_id": 0,
    "lat": 10.75,
    "lon": 106.67
}

# ======================
# PARSED INTENT + SLOT
# ======================
parsed = {
    "intent": "find_restaurant",
    "slots": {
        "food": "healthy, recommend món nhẹ",
        "price": 2,          # có thể None
        "distance_km": 5
    }
}
food = parsed["slots"].get("food")
# ======================
# CONNECT DB
# ======================
conn = psycopg2.connect(**DB_CONFIG)

# ======================
# STEP 1: BUILD SQL DYNAMIC
# ======================
def build_sql(parsed, user_context):

    user_lat = user_context["lat"]
    user_lon = user_context["lon"]

    distance_km = parsed["slots"].get("distance_km", 5)
    distance_threshold = distance_km / 111  # convert km → degree

    price = parsed["slots"].get("price")

    # 🔥 optional price
    if price is not None:
        price_condition = f"ABS(r.price_level - {price}) <= 1"
    else:
        price_condition = "TRUE"

    sql = f"""
    SELECT 
        r.id,
        r.name,
        r.price_level,
        r.latitude,
        r.longitude,

        SQRT(POWER(r.latitude - {user_lat}, 2) + 
             POWER(r.longitude - {user_lon}, 2)) AS distance,

        re.content,
        re.embedding,

        COALESCE(AVG(ur.rating), 0) as rating

    FROM restaurants r
    JOIN restaurant_embeddings re ON r.id = re.restaurant_id
    LEFT JOIN user_ratings ur ON r.id = ur.restaurant_id

    GROUP BY r.id, re.content, re.embedding

    HAVING 
        SQRT(POWER(r.latitude - {user_lat}, 2) + 
             POWER(r.longitude - {user_lon}, 2)) < {distance_threshold}
        AND {price_condition}

    LIMIT {TOP_N}
    """
    # sql = f"""
    # SELECT 
    #     r.id,
    #     r.name,
    #     r.price_level,
    #     r.latitude,
    #     r.longitude,

    #     SQRT(POWER(r.latitude - {user_lat}, 2) + 
    #         POWER(r.longitude - {user_lon}, 2)) AS distance,

    #     re.content,
    #     re.embedding,

    #     COALESCE(AVG(ur.rating), 0) as rating

    # FROM restaurants r
    # JOIN restaurant_embeddings re ON r.id = re.restaurant_id
    # LEFT JOIN user_ratings ur ON r.id = ur.restaurant_id

    # WHERE 
    #     SQRT(POWER(r.latitude - {user_lat}, 2) + 
    #         POWER(r.longitude - {user_lon}, 2)) < {distance_threshold}
    #     AND {price_condition}
    #     AND re.content ILIKE '%{food}%'

    # GROUP BY r.id, re.content, re.embedding

    # ORDER BY rating DESC
    # LIMIT {TOP_N}
    # """
    return sql

# ======================
# STEP 2: LOAD DATA
# ======================
sql_query = build_sql(parsed, user_context)
df = pd.read_sql(sql_query, conn)

df["embedding"] = df["embedding"].apply(lambda x: np.array(json.loads(x)))

print(f"✅ SQL returned {len(df)} candidates")

# ======================
# STEP 3: OPTIONAL KEYWORD FILTER (slot food)
# ======================


original_df = df.copy()



if food:
    filtered_df = df[df["content"].str.contains(food, case=False, na=False)]

    if len(filtered_df) > 0:
        print(f"✅ After food filter: {len(filtered_df)}")
        df = filtered_df
    else:
        print("⚠️ Không tìm thấy món → fallback semantic")
        df = original_df



# ======================
# STEP 4: TF-IDF
# ======================
if len(df) == 0:
    print("❌ No candidates → skip TF-IDF")
    tfidf_result = pd.DataFrame()
else:
    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(df["content"])

def search_tfidf(query):
    query_vec = tfidf.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    tmp = df.copy()
    tmp["score"] = scores

    return tmp.sort_values("score", ascending=False).head(TOP_K)

# ======================
# STEP 5: EMBEDDING
# ======================
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def search_embedding(query):
    query_emb = model.encode(query)

    scores = [
        cosine_similarity([query_emb], [doc_emb])[0][0]
        for doc_emb in df["embedding"]
    ]

    tmp = df.copy()
    tmp["score"] = scores

    return tmp.sort_values("score", ascending=False).head(TOP_K)

# ======================
# TEST
# ======================
query = "Mình đang ăn healthy, recommend món nhẹ"

tfidf_result = search_tfidf(query)
embedding_result = search_embedding(query)

# ======================
# PRINT
# ======================
print("\n===== TF-IDF RESULT =====")
print(tfidf_result[["name", "rating", "score"]])

print("\n===== EMBEDDING RESULT =====")
print(embedding_result[["name", "rating", "score"]])

C:\Users\Admin\AppData\Local\Temp\ipykernel_24336\1702454641.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql_query, conn)


✅ SQL returned 200 candidates
⚠️ Không tìm thấy món → fallback semantic

===== TF-IDF RESULT =====
                          name    rating     score
28              Quán Bột chiên  2.494355  0.141348
84   Quán Canh bí đỏ rong biển  2.731845  0.126376
68  Quán Thịt quay kho cà pháo  2.458333  0.125630
70          Quán Udon thịt xíu  3.054167  0.121480
30        Quán Lẩu ghẹ kim chi  2.680833  0.115071

===== EMBEDDING RESULT =====
                             name    rating     score
66         Quán Bê thui xào sa tế  2.797414  0.657656
174  Quán Chả Hoa Ngũ Sắc Phô Mai  2.859226  0.613546
126            Quán Gà kho rau củ  2.823925  0.610872
95              Quán Salad cá hồi  2.742816  0.608814
29    Quán Miến xào nghêu tay cầm  2.580645  0.596045


In [17]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json

def get_best_dish_semantic(restaurant_ids, food, model):

    # ===== LOAD MENU + EMBEDDING =====
    query = """
    SELECT restaurant_id, dish_name, embedding
    FROM menus
    WHERE restaurant_id = ANY(%s)
    """
    menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))

    # convert embedding string → vector
    menu_df["embedding"] = menu_df["embedding"].apply(
        lambda x: np.array(json.loads(x)) if isinstance(x, str) else np.array(x)
    )

    # ===== EMBEDDING QUERY =====
    query_emb = model.encode(food)

    # ===== TÍNH SIMILARITY =====
    menu_df["score"] = menu_df["embedding"].apply(
        lambda emb: cosine_similarity([query_emb], [emb])[0][0]
    )

    # ===== LẤY BEST MỖI QUÁN =====
    best = (
        menu_df
        .sort_values("score", ascending=False)
        .groupby("restaurant_id")
        .first()
    )

    return best[["dish_name", "score"]].to_dict(orient="index")

In [18]:
food = parsed["slots"].get("food")

best_dishes = get_best_dish_semantic(
    tfidf_result["id"].tolist(),
    food,
    model
)
print("\n===== TF-IDF RESULT =====")

for _, row in tfidf_result.iterrows():

    rid = row["id"]

    if rid in best_dishes:
        dish = best_dishes[rid]["dish_name"]
    else:
        dish = "Không tìm thấy món phù hợp"

    print(f"\n🍜 {row['name']} (⭐ {row['rating']:.1f})")
    print(f"👉 Món phù hợp: {dish}")
best_dishes_emb = get_best_dish_semantic(
    embedding_result["id"].tolist(),
    food,
    model
)

print("\n===== EMBEDDING RESULT =====")

for _, row in embedding_result.iterrows():

    rid = row["id"]

    if rid in best_dishes_emb:
        dish = best_dishes_emb[rid]["dish_name"]
    else:
        dish = "Không tìm thấy món phù hợp"

    print(f"\n🍜 {row['name']} (⭐ {row['rating']:.1f})")
    print(f"👉 Món phù hợp: {dish}")

C:\Users\Admin\AppData\Local\Temp\ipykernel_24336\1067276177.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))



===== TF-IDF RESULT =====

🍜 Quán Bột chiên (⭐ 2.5)
👉 Món phù hợp: Miến xào chay

🍜 Quán Canh bí đỏ rong biển (⭐ 2.7)
👉 Món phù hợp: Mực xào nấm hương

🍜 Quán Thịt quay kho cà pháo (⭐ 2.5)
👉 Món phù hợp: Salad cua thanh

🍜 Quán Udon thịt xíu (⭐ 3.1)
👉 Món phù hợp: Bánh đa cua

🍜 Quán Lẩu ghẹ kim chi (⭐ 2.7)
👉 Món phù hợp: Phở xào chua ngọt

===== EMBEDDING RESULT =====

🍜 Quán Bê thui xào sa tế (⭐ 2.8)
👉 Món phù hợp: Canh bó xôi bò viên

🍜 Quán Chả Hoa Ngũ Sắc Phô Mai (⭐ 2.9)
👉 Món phù hợp: Chả Hoa Ngũ Sắc Phô Mai

🍜 Quán Gà kho rau củ (⭐ 2.8)
👉 Món phù hợp: Gà kho rau củ

🍜 Quán Salad cá hồi (⭐ 2.7)
👉 Món phù hợp: Salad cá hồi

🍜 Quán Miến xào nghêu tay cầm (⭐ 2.6)
👉 Món phù hợp: Miến xào nghêu tay cầm


C:\Users\Admin\AppData\Local\Temp\ipykernel_24336\1067276177.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))


In [8]:
def get_menus(restaurant_ids):

    query = f"""
    SELECT restaurant_id, dish_name
    FROM menus
    WHERE restaurant_id = ANY(%s)
    """

    menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))

    # group thành list
    menu_grouped = menu_df.groupby("restaurant_id")["dish_name"].apply(list).to_dict()

    return menu_grouped

In [9]:
tfidf_result = search_tfidf(query)

menu_map = get_menus(tfidf_result["id"].tolist())

tfidf_result["menu"] = tfidf_result["id"].map(menu_map)

C:\Users\Admin\AppData\Local\Temp\ipykernel_24336\1255267676.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))


In [10]:
embedding_result = search_embedding(query)

menu_map2 = get_menus(embedding_result["id"].tolist())

embedding_result["menu"] = embedding_result["id"].map(menu_map2)

C:\Users\Admin\AppData\Local\Temp\ipykernel_24336\1255267676.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))


In [11]:
print("\n===== TF-IDF RESULT =====")
for _, row in tfidf_result.iterrows():
    print(f"\n🍜 {row['name']} (⭐ {row['rating']:.1f})")
    print("Menu:", ", ".join(row["menu"][:5]))

print("\n===== EMBEDDING RESULT =====")
for _, row in embedding_result.iterrows():
    print(f"\n🍜 {row['name']} (⭐ {row['rating']:.1f})")
    print("Menu:", ", ".join(row["menu"][:5]))


===== TF-IDF RESULT =====

🍜 Quán Bánh Tráng Trộn Ngon (⭐ 2.9)
Menu: Bánh Tráng Trộn Ngon, Chả Cá Lã Vọng, Khoai Môn Kẹp Thịt Chiên Giòn, Giả Khoai Tây Chiên Từ Bánh, Mứt Mận

🍜 Quán Rau Câu Bánh Trung Thu (⭐ 2.6)
Menu: Bò Nấu Bia, Bánh Cuốn Tôm, Bò Chiên Kiểu Thái, Rau Câu Bánh Trung Thu, Soup Gà

🍜 Quán Thịt Heo Giả Khô Bò (⭐ 2.9)
Menu: Thịt Heo Giả Khô Bò, Bò Kho Gừng, Xoài Lắc, Gà Sốt Chua Ngọt Tẩm Mè, Chả Ram Chiên Giòn

🍜 Quán Bánh Kẹp Tàn Ong (⭐ 2.6)
Menu: Thịt Ba Rọi Chiên Giòn, Hủ Tiếu Khô, Bánh Kẹp Tàn Ong, Bạch Tuộc Xào Cần Tỏi, Phở Cuốn Thịt Bò Rau Thơm

🍜 Quán Bún Bò Nam Bộ (⭐ 2.5)
Menu: Bún Bò Nam Bộ, Thịt Bò Xào Dưa Cải Chua, Bò Bít Tết Dầu Hào Maggi, Milo Đá Bào, Làm Bánh Bò Nướng

===== EMBEDDING RESULT =====

🍜 Quán Thịt Bò Kho Sả (⭐ 2.8)
Menu: Mì Tươi Trộn Sốt Bò Bằm, Rượu Trái Cây, Lòng Heo Nướng, Mực Dồn Thịt Rim Kẹo, Thịt Bò Kho Sả

🍜 Quán Thịt Bò Viên Thơm Ngon (⭐ 3.2)
Menu: Thịt Bò Viên Thơm Ngon, Bánh Chuối Bí Ngô Chiên Giòn, Bì Cuốn, Bánh Quy Bơ Hình Hoa Cúc,

In [5]:
def compare_models(tfidf_result, embedding_result):

    tfidf_ids = set(tfidf_result["id"])
    emb_ids = set(embedding_result["id"])

    intersection = tfidf_ids & emb_ids
    only_tfidf = tfidf_ids - emb_ids
    only_emb = emb_ids - tfidf_ids

    print("\n===== OVERLAP ANALYSIS =====")
    print(f"Common results: {len(intersection)}")
    print(f"Only TF-IDF: {len(only_tfidf)}")
    print(f"Only Embedding: {len(only_emb)}")

    print("\n👉 Common Restaurants:")
    print(df[df["id"].isin(intersection)][["name", "rating"]])

    print("\n👉 Only TF-IDF:")
    print(df[df["id"].isin(only_tfidf)][["name", "rating"]])

    print("\n👉 Only Embedding:")
    print(df[df["id"].isin(only_emb)][["name", "rating"]])

In [6]:
compare_models(tfidf_result, embedding_result)


===== OVERLAP ANALYSIS =====
Common results: 0
Only TF-IDF: 5
Only Embedding: 5

👉 Common Restaurants:
Empty DataFrame
Columns: [name, rating]
Index: []

👉 Only TF-IDF:
                               name    rating
26  Quán Miến đậu xanh xào thập cẩm  2.669345
34                 Quán Bánh há cảo  2.920139
40                     Quán Phở xào  2.696839
48             Quán Lẩu ghẹ kim chi  2.680833
90                    Quán Phở trộn  3.037222

👉 Only Embedding:
                           name    rating
11             Quán Heo kho tàu  2.754630
33             Quán Bò kho chay  2.128333
43    Quán Sườn non xốt giấm đỏ  2.543210
46  Quán Miến xào nghêu tay cầm  2.580645
59        Quán Gà nướng nồi đất  2.310897


In [7]:
def evaluate_single_result(result, user_data, k=5):

    relevance = []

    for rid in result["id"]:
        rel = user_data[user_data["restaurant_id"] == rid]["relevant"]
        relevance.append(int(rel.values[0]) if len(rel) > 0 else 0)

    precision = sum(relevance[:k]) / k

    total_relevant = user_data["relevant"].sum()
    recall = sum(relevance[:k]) / total_relevant if total_relevant > 0 else 0

    # NDCG
    def dcg(rel):
        return sum([rel[i] / np.log2(i+2) for i in range(len(rel))])

    dcg_val = dcg(relevance[:k])
    idcg_val = dcg(sorted(relevance, reverse=True)[:k])
    ndcg = dcg_val / idcg_val if idcg_val > 0 else 0

    return precision, recall, ndcg

In [8]:
def compare_metrics(tfidf_result, embedding_result, user_data):

    p1, r1, n1 = evaluate_single_result(tfidf_result, user_data)
    p2, r2, n2 = evaluate_single_result(embedding_result, user_data)

    print("\n===== METRICS COMPARISON =====")
    print(f"{'Model':<15}{'Precision':<10}{'Recall':<10}{'NDCG':<10}")
    print(f"{'TF-IDF':<15}{p1:.3f}{r1:.3f}{n1:.3f}")
    print(f"{'Embedding':<15}{p2:.3f}{r2:.3f}{n2:.3f}")